# Notebook 17 — Đối chứng mã hóa hai host (chỉ nhánh A)

Cùng dữ liệu Notebook 16; thay `x_i,0, x_i,1` bằng `1-z_i, z_i`, giữ y và slack CPU/RAM.
Đọc kết quả cũ từ ZIP, chạy **45 lượt compact + 1 smoke vô nghiệm**. Không chạy lại Notebook 16.

**Đọc kết quả đúng:** giảm qubit không bảo đảm tăng chất lượng. Uniform của mã hóa mới cũng tăng P_opt 16 lần.
So sánh P_opt và lift so với uniform riêng; không suy ra quantum advantage từ tỷ lệ thành công.
Hamiltonian được chuẩn hóa max_abs riêng, trạng thái đầu khác giữa hai mã hóa.

Chọn kernel `nt409-khiem`, Restart Kernel trước phiên đo. Chạy tuần tự, không chạy notebook nặng khác cùng lúc.
Xem `docs/two_host_encoding_protocol.md` và `README_BRANCH_A_NOTEBOOK17.md`.


## 1. Đường dẫn và lựa chọn phiên

Lần đầu giữ `RESUME_DIR = None`. Bước 2 in đường dẫn: chép vào biến này khi tiếp tục phiên cũ.
`INSTANCE_IDS = None` chạy cả ba instance; hoặc dùng `['p1_choice']`, `['p1_tight']`, `['p0_reference']` để chia phiên.
Không đổi config, dữ liệu hoặc seed giữa campaign. Giữ `RETRY_FAILED=False`; chỉ bật sau khi xử lý lỗi tài nguyên/môi trường.


In [1]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/benchmark_a_campaign.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook từ bên trong thư mục nt409-task-placement.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
BASELINE_ZIP = ROOT / 'data/benchmarks/nt409_encoding_v1/baseline_notebook16.zip'
RESUME_DIR = None  # Ví dụ: 'data/reference_results/phase4i_encoding/encoding_...'
INSTANCE_IDS = None
RETRY_FAILED = False
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.benchmark_a_campaign import load_benchmark
from src.two_host_campaign import (open_campaign, import_baseline, prepare_models,
    run_preflights, run_campaign, verify_campaign, result_rows, export_campaign)
context = load_benchmark(ROOT)
print('ROOT:', ROOT)
print('Python:', sys.version)


ROOT: /workspace/scratch/5e8f9a5e7776/nt409-task-placement
Python: 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]


## 2. Audit ZIP Notebook 16 và mở campaign

Kiểm tra hash, ma trận, dữ liệu, counts, decoder, năng lượng và metrics của báo cáo cũ.
Lần đầu audit có thể mất vài phút; tiến độ hiện mỗi 10 attempts. Các lần dùng lại trong cùng kernel có cache theo SHA-256.
Nếu runtime khác, vẫn so sánh chất lượng theo cùng giao thức; không kết luận tốc độ giữa hai môi trường.


In [2]:
FOLDER, campaign = open_campaign(ROOT, context, BASELINE_ZIP, RESUME_DIR)
baseline = import_baseline(FOLDER / 'baseline_notebook16.zip', context)
print('RESUME_DIR =', repr(FOLDER.relative_to(ROOT).as_posix()))
print('Baseline campaign:', baseline['campaign_id'])
print('Runtime ghi nhận trùng nhau:', baseline['runtime_matches_current'])
print('Cùng runtime chưa chứng minh cùng tải máy/CPU; thời gian cần diễn giải thận trọng.')
print('Số lượt mới theo kế hoạch:', len(campaign['planned_runs']))


Audit Notebook16: 1/55 attempts
Audit Notebook16: 10/55 attempts
Audit Notebook16: 20/55 attempts
Audit Notebook16: 30/55 attempts
Audit Notebook16: 40/55 attempts
Audit Notebook16: 50/55 attempts
RESUME_DIR = 'data/reference_results/phase4i_encoding/encoding_20261010T155454118713Z'
Baseline campaign: 20261010T081035000606Z
Runtime ghi nhận trùng nhau: False
Cùng runtime chưa chứng minh cùng tải máy/CPU; thời gian cần diễn giải thận trọng.
Số lượt mới theo kế hoạch: 46


## 3. Chứng nhận mô hình và đối chứng riêng

Kiểm tra đồng nhất đa thức năng lượng trên không gian one-hot và toàn bộ tập placement.
Các phép liệt kê dưới đây phục vụ kiểm chứng mô hình, không được báo cáo là thời gian giải QAOA.


In [3]:
prepared = prepare_models(FOLDER, context)
model_rows = []
for item in prepared:
    ref = item['reference']; cert = item['certificate']; uniform = ref['uniform_baseline']
    model_rows.append(dict(instance=item['instance_id'], role=item['role'],
        old_qubits=cert['old_qubits'], compact_qubits=cert['new_qubits'],
        optimal_hosts=ref['original_best_objective'], qubo_minimum=ref['minimum_energy'],
        uniform_feasible=uniform['feasible_probability'],
        uniform_optimal=uniform['optimal_placement_probability'],
        preprocessing_seconds=item['model_reference_certificate_seconds']))
display(pd.DataFrame(model_rows))


     instance             role  old_qubits  compact_qubits  optimal_hosts  qubo_minimum  uniform_feasible  uniform_optimal  preprocessing_seconds
    p1_choice        benchmark          20              16            1.0           1.0          0.187500         0.015625               0.181722
     p1_tight        benchmark          18              14            2.0           2.0          0.062500         0.062500               0.158943
 p0_reference        benchmark          23              19            2.0           2.0          0.015625         0.015625               0.216177
p1_infeasible negative_control          20              16            NaN           8.0          0.000000              NaN               0.198594


## 4. Probe tài nguyên tại mọi độ sâu đã chọn

Ba điểm góc có mọi layer hoạt động cho mỗi instance/p, kèm đo batch cuối và giải mã.
Dự báo thời gian dùng hệ số 1,5; đây không phải bảo đảm thời gian hay hard cap RAM của hệ điều hành.
Chỉ chạy phần tối ưu khi mọi probe đã chọn có trạng thái `passed`.


In [4]:
campaign = run_preflights(FOLDER, context, INSTANCE_IDS, retry_failed=RETRY_FAILED)
display(pd.DataFrame([a for a in campaign['attempts'] if a['kind'] == 'probe'])[
    ['instance_id', 'reps', 'status', 'report']])


Compact probe p1_choice p=1
passed
Compact probe p1_choice p=2
passed
Compact probe p1_choice p=3
passed
Compact probe p1_tight p=1
passed
Compact probe p1_tight p=2
passed
Compact probe p1_tight p=3
passed
Compact probe p0_reference p=1
passed
Compact probe p0_reference p=2
passed
Compact probe p0_reference p=3
passed
 instance_id  reps status                                report
   p1_choice     1 passed attempts/0001_probe_ad9007c19cb7.json
   p1_choice     2 passed attempts/0002_probe_da713938808d.json
   p1_choice     3 passed attempts/0003_probe_4be889a14008.json
    p1_tight     1 passed attempts/0004_probe_3017deb29acb.json
    p1_tight     2 passed attempts/0005_probe_395787249947.json
    p1_tight     3 passed attempts/0006_probe_ade198bdae9f.json
p0_reference     1 passed attempts/0007_probe_c9b813796568.json
p0_reference     2 passed attempts/0008_probe_d49d0ba85aa1.json
p0_reference     3 passed attempts/0009_probe_ed6bcd449db8.json


## 5. Chạy QAOA compact

Tự lưu từng attempt, giữ cả failure. Nếu ngắt kernel, mở lại cùng `RESUME_DIR`.
Lượt `completed` không được chạy lại; hội tụ số học hoặc chạm cap đều được giữ nếu có đủ mẫu cuối.
Nếu bước này báo lỗi preflight, chuyển tới bước 8 để xuất dữ liệu partial.


In [5]:
campaign = run_campaign(FOLDER, context, INSTANCE_IDS, retry_failed=RETRY_FAILED, progress=True)


Compact p1_infeasible p=1 seed=70
  eval 1: E=493.671875, 0.1s
completed
Compact p1_choice p=1 seed=70
  eval 1: E=289.303711, 0.1s
  eval 10: E=146.544922, 0.4s
  eval 20: E=134.189453, 0.7s
  eval 30: E=129.856445, 1.0s
completed
Compact p1_choice p=1 seed=71
  eval 1: E=285.156250, 0.1s
  eval 10: E=165.574219, 0.4s
  eval 20: E=157.997070, 0.7s
  eval 30: E=155.538086, 1.0s
completed
Compact p1_choice p=1 seed=72
  eval 1: E=291.465820, 0.1s
  eval 10: E=173.204102, 0.4s
  eval 20: E=172.291992, 0.8s
completed
Compact p1_choice p=1 seed=73
  eval 1: E=286.313477, 0.1s
  eval 10: E=162.208008, 0.4s
  eval 20: E=152.610352, 0.7s
completed
Compact p1_choice p=1 seed=74
  eval 1: E=293.946289, 0.1s
  eval 10: E=170.456055, 0.4s
  eval 20: E=167.240234, 0.8s
completed
Compact p1_choice p=2 seed=70
  eval 1: E=289.303711, 0.1s
  eval 10: E=76.627930, 0.4s
  eval 20: E=51.197266, 0.8s
  eval 30: E=55.810547, 1.2s
  eval 40: E=52.056641, 1.6s
completed
Compact p1_choice p=2 seed=71
  eval 

## 6. Audit kết quả, giữ trạng thái pending/failure

Bảng so sánh chỉ gộp 45 lượt chính; smoke vô nghiệm hiển thị riêng.
`total_algorithm_shots` gồm training, batch eigenstate nội bộ của Qiskit và final batch độc lập;
không gồm probe. Với baseline, giá trị được tái dựng theo Qiskit Optimization 0.7.0; compact đếm các lần gửi sampler.


In [6]:
campaign = json.loads((FOLDER / 'campaign.json').read_text(encoding='utf-8'))
completed = verify_campaign(FOLDER, campaign, context)
compact_rows = result_rows(FOLDER, campaign)
comparison = pd.DataFrame(baseline['rows'] + compact_rows)
formal = comparison[comparison.role == 'benchmark'].copy()
compact_formal = formal[formal.encoding == 'two_host_binary_v1']
COMPLETE = len(compact_formal) == 45 and compact_formal.status.eq('completed').all()
print('Trạng thái:', 'COMPLETE — dữ liệu, chưa phải chứng minh hiệu quả' if COMPLETE else 'PARTIAL')
display(compact_formal[['instance_id','p','seed','status','optimal_probability','optimal_lift','nfev','total_algorithm_shots']])
display(comparison[comparison.role == 'negative_control'][['encoding','status','found_valid','optimal_probability']])
summary = (formal.groupby(['encoding','instance_id','p'], sort=False)
    .agg(planned=('status','size'), completed=('status', lambda s: s.eq('completed').sum()),
         optimal_probability_mean=('optimal_probability','mean'),
         optimal_probability_sd=('optimal_probability','std'),
         optimal_lift_mean=('optimal_lift','mean'),
         feasible_probability_mean=('feasible_probability','mean'),
         found_optimal_runs=('found_optimal', lambda s: s.fillna(False).sum()),
         total_seconds_mean=('total_seconds','mean'))).reset_index()
print('Mean/SD chỉ trên completed; xem cột completed/planned. Không biến pending thành P_opt=0.')
display(summary)


Trạng thái: COMPLETE — dữ liệu, chưa phải chứng minh hiệu quả
 instance_id  p  seed    status  optimal_probability  optimal_lift  nfev  total_algorithm_shots
   p1_choice  1    70 completed             0.025391      1.625000    31                  33792
   p1_choice  1    71 completed             0.018555      1.187500    31                  33792
   p1_choice  1    72 completed             0.037109      2.375000    28                  30720
   p1_choice  1    73 completed             0.018555      1.187500    28                  30720
   p1_choice  1    74 completed             0.027344      1.750000    25                  27648
   p1_choice  2    70 completed             0.012695      0.812500    47                  50176
   p1_choice  2    71 completed             0.026367      1.687500    44                  47104
   p1_choice  2    72 completed             0.020508      1.312500    41                  44032
   p1_choice  2    73 completed             0.029297      1.875000    42  

## 7. Biểu đồ P_opt và lift trên cùng instance

Chỉ vẽ khi đủ 45 lượt compact. Error bar là độ lệch chuẩn giữa năm seed, không phải khoảng tin cậy.
Lift = P_opt / P_opt(uniform riêng của mã hóa); đường lift=1 là uniform.
Chỉ ba instance đã biết, chưa đánh giá khái quát hóa hay scaling lên 8 task/4 host.


In [7]:
if COMPLETE:
    fig, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
    for col, instance in enumerate(['p1_choice', 'p1_tight', 'p0_reference']):
        for encoding, label in [('one_hot_slack_v1_1','Notebook 16'), ('two_host_binary_v1','Compact')]:
            data = formal[(formal.instance_id == instance) & (formal.encoding == encoding)]
            grouped = data.groupby('p')
            for row, metric in enumerate(['optimal_probability','optimal_lift']):
                means = grouped[metric].mean(); sd = grouped[metric].std(); factor = 100 if row == 0 else 1
                axes[row,col].errorbar(means.index, means*factor, yerr=sd*factor,
                    marker='o', capsize=3, label=label)
        axes[0,col].set_title(instance)
        axes[0,col].set_ylabel('Optimal shots (%)')
        axes[1,col].set_ylabel('Lift over own uniform')
        axes[1,col].axhline(1, color='gray', linestyle='--')
        for ax in axes[:,col]:
            ax.set_xticks([1,2,3]); ax.set_xlabel('p'); ax.grid(alpha=.2)
    axes[0,0].legend(); fig.savefig(FOLDER / 'encoding_quality.png', dpi=160)
    plt.show()
else:
    print('Chưa đủ dữ liệu; giữ bảng trạng thái, chưa vẽ đường so sánh tổng kết.')


## 8. Xuất gói để review

Gói ZIP chứa báo cáo mới, ZIP baseline nguyên vẹn, CSV so sánh, manifest và checksum.
Lưu notebook đã chạy rồi gửi ZIP được in dưới đây cùng notebook executed.
Kết luận chất lượng chỉ sau khi review; không tăng p hoặc chọn lại seed dựa trên biểu đồ.


In [8]:
archive, exported_rows = export_campaign(FOLDER, context)
print('Gửi ZIP:', archive)
print('Kèm notebook executed đã Save.')


Gửi ZIP: /workspace/scratch/5e8f9a5e7776/nt409-task-placement/data/reference_results/phase4i_encoding/encoding_20261010T155454118713Z/encoding_20261010T155454118713Z.zip
Kèm notebook executed đã Save.
